# LangChain Decoded: Building Modular LLM Applications

### LangChain Demo
This notebook demonstrates LangChain concepts without requiring API keys, using FakeListLLM and HuggingFace embeddings.


### Install dependencies

In [ ]:
# Install dependencies (run once)
!pip install langchain langchain-core langchain-community faiss-cpu chromadb tiktoken transformers
!pip install -U langchain langchain-community langchain-core transformers langchain-huggingface huggingface_hub

### Imports
We use `langchain_core` for prompts and runnables, `langchain_community` for integrations, and HuggingFace for embeddings.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.fake import FakeListLLM
from langchain_core.runnables import RunnableSequence
from langchain_core.tools import Tool
from langchain.agents import initialize_agent
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

### Fake LLM Demo
We use `FakeListLLM` to simulate responses offline.

In [ ]:
llm = FakeListLLM(responses=["This is a demo response from a fake LLM."])
print(llm.invoke("Explain LangChain in one sentence."))

### 1. INTRODUCTION 

Large Language Models are powerful but lack structure when building any real-world applications. LangChain provides a framework that helps developers connect the LLMs with tools, memory, and workflows, making it easier to build intelligent systems.

In [ ]:
response = llm.invoke("What is LangChain?")
print(response)

### 2. CORE COMPONENTS

#### Prompt Templates

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} clearly"
)

#### Chains

In [ ]:
from langchain.chains import LLMChain

chain = LLMChain(llm=llm, prompt=template)

print(chain.invoke({"topic": "AI Agents"}))

#### Memory

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory
)

print(conversation.predict(input="Hi, my name is Emma"))
print(conversation.predict(input="What is my name?"))

#### Tools

In [ ]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    return a * b

print(multiply.run({"a": 3, "b": 4}))

#### Agents

In [ ]:
from langchain.agents import initialize_agent, AgentType

tools = [multiply]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

print(agent.run("Multiply 5 and 6"))

#### Document Loader + Vector Store
We load a text file, embed it with HuggingFace, and query with FAISS.

In [ ]:
with open("sample.txt", "w") as f:
    f.write("LangChain is a framework for building applications with large language models.")

loader = TextLoader("sample.txt")
docs = loader.load()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(docs, embeddings)

query = "What is LangChain?"
results = db.similarity_search(query)
print(results[0].page_content)